# Kaggle: Predicción de precios de portátiles (Random Forest)
Notebook de referencia con **preprocesado detallado** + **RandomForestRegressor** + validación + generación de `submission.csv`.

✅ **Métrica** en Kaggle: **RMSE** (cuanto más bajo, mejor).
✅ **Flujo**: `train.csv` (entrenar) → validación local → `test.csv` (predecir Kaggle) → `submission.csv`.


In [21]:
import numpy as np
import pandas as pd

import re
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Para el chequeador (del notebook original)
import urllib.request
from PIL import Image

RANDOM_STATE = 42


## 1) Carga de datos

In [14]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')

train.shape, test.shape, sample.shape

((912, 13), (391, 12), (391, 2))

In [15]:
train.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


## 2) Preprocesado (limpieza + feature engineering)
Pasamos de strings “sucios” a features útiles:
- `Ram`: `'8GB' → 8`
- `Weight`: `'1.86kg' → 1.86`
- `ScreenResolution`: `X_res`, `Y_res`, `ppi`, `Touchscreen`, `IPS`
- `Cpu`: `Cpu_brand`, `Cpu_family`, `Cpu_ghz`
- `Memory`: `SSD_GB`, `HDD_GB`, `Flash_GB`, `Hybrid_GB`, `Total_Memory_GB`
- `Gpu`: `Gpu_brand`


In [16]:
def _to_gb(value: str) -> float:
    """Convierte strings tipo '1TB' o '256GB' a GB numérico."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    m = re.search(r'(\d+(?:\.\d+)?)\s*(TB|GB)', value, flags=re.I)
    if not m:
        return np.nan
    num = float(m.group(1))
    unit = m.group(2).upper()
    return num * 1024 if unit == 'TB' else num


def clean_ram(s):
    # '8GB' -> 8
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan


def clean_weight(s):
    # '1.86kg' -> 1.86
    if pd.isna(s):
        return np.nan
    s = str(s).lower().replace('kg', '').strip()
    try:
        return float(s)
    except:
        return np.nan


def parse_screen_resolution(s):
    """Devuelve: X_res, Y_res, IPS, Touchscreen"""
    if pd.isna(s):
        return pd.Series([np.nan, np.nan, 0, 0], index=['X_res', 'Y_res', 'IPS', 'Touchscreen'])
    s = str(s)

    ips = 1 if re.search(r'\bIPS\b', s, flags=re.I) else 0
    touch = 1 if re.search(r'touch', s, flags=re.I) else 0

    m = re.search(r'(\d{3,4})\s*x\s*(\d{3,4})', s)
    if not m:
        return pd.Series([np.nan, np.nan, ips, touch], index=['X_res', 'Y_res', 'IPS', 'Touchscreen'])
    x = int(m.group(1))
    y = int(m.group(2))
    return pd.Series([x, y, ips, touch], index=['X_res', 'Y_res', 'IPS', 'Touchscreen'])


def compute_ppi(inches, x_res, y_res):
    if pd.isna(inches) or pd.isna(x_res) or pd.isna(y_res):
        return np.nan
    return ((x_res**2 + y_res**2) ** 0.5) / inches


def parse_cpu(s):
    """Devuelve: Cpu_brand, Cpu_family, Cpu_ghz"""
    if pd.isna(s):
        return pd.Series([np.nan, np.nan, np.nan], index=['Cpu_brand', 'Cpu_family', 'Cpu_ghz'])
    s = str(s)

    brand = 'Intel' if 'intel' in s.lower() else ('AMD' if 'amd' in s.lower() else 'Other')

    fam_patterns = [
        r'(Core i[3579])',
        r'(Ryzen\s*[3579])',
        r'(Celeron)',
        r'(Pentium)',
        r'(Atom)',
        r'(Xeon)',
        r'(M[357])'
    ]
    family = None
    for pat in fam_patterns:
        m = re.search(pat, s, flags=re.I)
        if m:
            family = m.group(1)
            break
    if family is None:
        tokens = s.split()
        family = " ".join(tokens[:2]) if len(tokens) >= 2 else tokens[0]

    m = re.search(r'(\d+(?:\.\d+)?)\s*GHz', s, flags=re.I)
    ghz = float(m.group(1)) if m else np.nan

    return pd.Series([brand, family, ghz], index=['Cpu_brand', 'Cpu_family', 'Cpu_ghz'])


def parse_memory(s):
    """Extrae tamaños SSD/HDD/Flash/Hybrid y total en GB."""
    if pd.isna(s):
        return pd.Series([0, 0, 0, 0, 0], index=['SSD_GB','HDD_GB','Flash_GB','Hybrid_GB','Total_Memory_GB'])

    s = str(s).replace(' ', '')
    parts = s.split('+')

    ssd = hdd = flash = hybrid = 0.0

    for p in parts:
        gb = _to_gb(p)
        if np.isnan(gb):
            continue
        p_low = p.lower()
        if 'ssd' in p_low:
            ssd += gb
        elif 'hdd' in p_low:
            hdd += gb
        elif 'flash' in p_low:
            flash += gb
        elif 'hybrid' in p_low:
            hybrid += gb
        else:
            flash += gb

    total = ssd + hdd + flash + hybrid
    return pd.Series([ssd, hdd, flash, hybrid, total],
                     index=['SSD_GB','HDD_GB','Flash_GB','Hybrid_GB','Total_Memory_GB'])


def parse_gpu_brand(s):
    if pd.isna(s):
        return np.nan
    s = str(s).lower()
    if 'nvidia' in s:
        return 'Nvidia'
    if 'amd' in s or 'radeon' in s:
        return 'AMD'
    if 'intel' in s:
        return 'Intel'
    return 'Other'


def preprocess(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['Ram_GB'] = df['Ram'].apply(clean_ram)
    df['Weight_kg'] = df['Weight'].apply(clean_weight)

    sr = df['ScreenResolution'].apply(parse_screen_resolution)
    df = pd.concat([df, sr], axis=1)
    df['ppi'] = df.apply(lambda r: compute_ppi(r['Inches'], r['X_res'], r['Y_res']), axis=1)

    cpu = df['Cpu'].apply(parse_cpu)
    df = pd.concat([df, cpu], axis=1)

    mem = df['Memory'].apply(parse_memory)
    df = pd.concat([df, mem], axis=1)

    df['Gpu_brand'] = df['Gpu'].apply(parse_gpu_brand)

    # Columnas originales demasiado sucias o demasiado granulares
    drop_cols = ['Ram', 'Weight', 'ScreenResolution', 'Cpu', 'Memory', 'Gpu', 'Product']
    df = df.drop(columns=drop_cols)

    return df


In [17]:
train_p = preprocess(train)
test_p = preprocess(test)

train_p.head()

,laptop_ID,Company,TypeName,Inches,OpSys,Price_in_euros,Ram_GB,Weight_kg,X_res,Y_res,...,ppi,Cpu_brand,Cpu_family,Cpu_ghz,SSD_GB,HDD_GB,Flash_GB,Hybrid_GB,Total_Memory_GB,Gpu_brand
0,755,HP,Notebook,15.6,Windows 10,539.00,8.0,1.86,1920,1080,...,141.211998,Intel,Core i3,2.0,256.0,0.0,0.0,0.0,256.0,Intel
1,618,Dell,Gaming,15.6,Windows 10,879.01,16.0,2.59,1920,1080,...,141.211998,Intel,Core i7,2.6,0.0,1024.0,0.0,0.0,1024.0,Nvidia
2,909,HP,Notebook,15.6,Windows 10,900.00,8.0,2.04,1920,1080,...,141.211998,Intel,Core i7,2.7,0.0,1024.0,0.0,0.0,1024.0,Nvidia
3,2,Apple,Ultrabook,13.3,macOS,898.94,8.0,1.34,1440,900,...,127.677940,Intel,Core i5,1.8,0.0,0.0,128.0,0.0,128.0,Intel
4,286,Dell,Notebook,15.6,Linux,428.00,4.0,2.25,1920,1080,...,141.211998,Intel,Core i3,2.0,0.0,1024.0,0.0,0.0,1024.0,AMD


### 2.1) Nulos tras el preprocesado
Random Forest no acepta NaNs: imputamos con **mediana** (numéricas) y **moda** (categóricas) calculadas SOLO en train.
Luego hacemos **one-hot** y alineamos columnas train/test.

In [18]:
# Separar target
y = train_p['Price_in_euros']
X = train_p.drop(columns=['Price_in_euros'])

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','bool']).columns.tolist()

print('Num cols:', num_cols)
print('Cat cols:', cat_cols)

Num cols: ['laptop_ID', 'Inches', 'Ram_GB', 'Weight_kg', 'X_res', 'Y_res', 'IPS', 'Touchscreen', 'ppi', 'Cpu_ghz', 'SSD_GB', 'HDD_GB', 'Flash_GB', 'Hybrid_GB', 'Total_Memory_GB']
Cat cols: ['Company', 'TypeName', 'OpSys', 'Cpu_brand', 'Cpu_family', 'Gpu_brand']


In [19]:
def impute_and_encode(train_X, test_X, num_cols, cat_cols):
    train_X = train_X.copy()
    test_X = test_X.copy()

    # Numéricas: mediana
    med = train_X[num_cols].median()
    train_X[num_cols] = train_X[num_cols].fillna(med)
    test_X[num_cols] = test_X[num_cols].fillna(med)

    # Categóricas: moda
    for c in cat_cols:
        mode_val = train_X[c].mode(dropna=True)[0]
        train_X[c] = train_X[c].fillna(mode_val)
        test_X[c] = test_X[c].fillna(mode_val)

    # One-hot
    train_X = pd.get_dummies(train_X, columns=cat_cols, drop_first=True)
    test_X = pd.get_dummies(test_X, columns=cat_cols, drop_first=True)

    # Alinear columnas
    test_X = test_X.reindex(columns=train_X.columns, fill_value=0)

    return train_X, test_X

X_encoded, test_encoded = impute_and_encode(X, test_p, num_cols, cat_cols)

X_encoded.shape, test_encoded.shape

((912, 68), (391, 68))

## 3) Validación local (train vs validación)
Mantenemos una validación simple con `train_test_split` para que puedas comparar manualmente.
Métrica: **RMSE**.

In [23]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=RANDOM_STATE
)

rf = RandomForestRegressor(
    n_estimators=600,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_train = rf.predict(X_train)
pred_val = rf.predict(X_val)

mse_train = mean_squared_error(y_train, pred_train)
mse_val   = mean_squared_error(y_val, pred_val)

rmse_train = np.sqrt(mse_train)
rmse_val   = np.sqrt(mse_val)

print(f"RMSE train: {rmse_train:.3f}")
print(f"RMSE val:   {rmse_val:.3f}")
print(f"Diferencia: {rmse_val - rmse_train:.3f}")

RMSE train: 104.735
RMSE val:   335.499
Diferencia: 230.764


## 4) Entrenamiento final (con todo el train) + predicción en test Kaggle

In [24]:
rf_final = RandomForestRegressor(
    n_estimators=800,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_final.fit(X_encoded, y)

test_preds = rf_final.predict(test_encoded)

submission = pd.DataFrame({
    'laptop_ID': test_p['laptop_ID'].values,
    'Price_in_euros': test_preds
})

submission.head(), submission.shape

(   laptop_ID  Price_in_euros
 0        209     1495.032087
 1       1281      295.858287
 2       1168      390.634275
 3       1231     1001.613162
 4       1020     1109.030950,
 (391, 2))

## 5) Chequeador de Kaggle (mismo que el notebook del profe)
Si pasa, guarda automáticamente `submission.csv` listo para subir.

In [25]:
def chequeador(df_to_submit):
    """
    Esta función se asegura de que tu submission tenga la forma requerida por Kaggle.

    Si es así, se guardará el dataframe en un `csv` y estará listo para subir a Kaggle.

    Si no, LEE EL MENSAJE Y HAZLE CASO.

    Si aún no:
    - apaga tu ordenador,
    - date una vuelta,
    - enciendelo otra vez,
    - abre este notebook y
    - leelo todo de nuevo.
    Todos nos merecemos una segunda oportunidad. También tú.
    """
    if df_to_submit.shape == sample.shape:
        if df_to_submit.columns.all() == sample.columns.all():
            if df_to_submit.laptop_ID.all() == sample.laptop_ID.all():
                print("You're ready to submit!")
                df_to_submit.to_csv("submission.csv", index=False)  # MUY importante index=False
                urllib.request.urlretrieve(
                    "https://www.mihaileric.com/static/evaluation-meme-e0a350f278a36346e6d46b139b1d0da0-ed51e.jpg",
                    "gfg.png"
                )
                img = Image.open("gfg.png")
                img.show()
            else:
                print("Check the ids and try again")
        else:
            print("Check the names of the columns and try again")
    else:
        print("Check the number of rows and/or columns and try again")
        print("\nMensaje secreto del TA: No me puedo creer que después de todo este notebook hayas hecho algún cambio en las filas de `test.csv`. Lloro.")

In [26]:
chequeador(submission)

You're ready to submit!
